# Pre-entrega 6: Orquestador Multi-Agente — Demo del flujo de delegación

Este notebook usa el código del repo (`state.py`, `agents/`, `graph.py`). Correrlo desde la raíz del repo, con el `.env` completo y `chunks.json` copiado de la Pre-entrega 4.

## 1. Construir el grafo

In [ ]:
from graph import construir_grafo
from llm import crear_llm, texto

app = construir_grafo(crear_llm())

## 2. Visualizar el grafo

Supervisor en el centro; los especialistas siempre vuelven a él. Las flechas punteadas son las aristas condicionales.

In [ ]:
from IPython.display import Image, display

display(Image(app.get_graph().draw_mermaid_png()))

## 3. Consulta que requiere los dos dominios

Para responderla hace falta **investigar** (qué campos tiene el modelo `Item` según la documentación) y **analizar** (validar los JSON contra ese esquema). Los datos del esquema no están en la pregunta: el supervisor está obligado a pasar primero por el investigador.

In [ ]:
from main import CONSULTA_EJEMPLO, ICONOS, estado_inicial
from config import RECURSION_LIMIT

print(CONSULTA_EJEMPLO)

## 4. Flujo de delegación paso a paso

In [ ]:
estado_final = None
async for estado in app.astream(estado_inicial(CONSULTA_EJEMPLO), {"recursion_limit": RECURSION_LIMIT}, stream_mode="values"):
    estado_final = estado
    ultimo = estado["messages"][-1]
    if getattr(ultimo, "name", None) in ICONOS:
        print(f"{ICONOS[ultimo.name]} [{ultimo.name}]\n{texto(ultimo)}\n")

## 5. Estado compartido: quién aportó qué

In [ ]:
for c in estado_final["contribuciones"]:
    print(f"--- paso {c['paso']} · {c['agente']} ---")
    print(f"Tarea: {c['instruccion']}")
    print(f"Aporte: {c['aporte']}\n")

print("pasos:", estado_final["pasos"], "| task_completed:", estado_final["task_completed"])

## 6. Respuesta final

In [ ]:
print(texto(estado_final["messages"][-1]))